# Consolidated grounding-judge check
Run all four code cells after Drive sync. A100 GPU is recommended if available; T4 is a slower option.
This compares the existing Qwen3 4B checkpoint and Qwen2.5 14B under one simplified workflow: 20 examples, separate statement and citation tasks, 36 calls per model (72 total). Both use four-bit inference and load sequentially. The larger checkpoint is a candidate, not a guaranteed improvement.
Eight examples were used before; twelve are new to these runs. All are assistant-authored synthetic checks, not independent human validation. Original and new results are reported separately.
Python supplies numerical relations. Judges return verdicts, categorical reason codes and evidence IDs rather than long rationales. Diagnosis C/D outputs remain unchanged. No PHM explanation scoring runs automatically.
The fourteen-billion-parameter checkpoint requires an additional model download. Cached attempts resume without automatic retries. Send the one final Latest report.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Disconnect and delete runtime, reconnect, then complete Google authorisation. No inference has started.') from exc
if not (ROOT/'Files/phm_judge_v2.py').is_file():
    raise FileNotFoundError(f'Wait for Drive sync or correct the project path: {ROOT}. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_judge_v2.py': 'f1e913c7fdbb0ff6ee65c47a9ed4c9a14107383008692a47a5ac0a33167dce50', 'test_phm_judge_v2.py': '96eea9f4ceb19cefe1d427418eca5d4a8009e24f2edc47f7c39955eaa14d5197', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'requirements-qwen-colab.txt': '0290713affb7b64776db6d09da7735ef3abbf12c38cf511c473abb0ec76f4226', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b'}
PACK_HASHES = {'inputs.jsonl': '1f3b8eaa848500e898fdecc489277bbfd46b30d6fcd21254fd77edd9b875ddc6', 'answers_DO_NOT_SEND_TO_JUDGE.jsonl': '657b0828da9310b164e8b5b215bc36d965615b5cde794394aee4c672732eed85'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256((ROOT/'Files'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
for name, expected in PACK_HASHES.items():
    assert hashlib.sha256((ROOT/'Outputs/reviews/grounding_v2'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
print('Code and frozen calibration inputs verified.')


Mounted at /content/drive
Code and frozen calibration inputs verified.


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-qwen-colab.txt', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed. Two public Qwen checkpoints; no Llama approval required.')


Dependencies installed. Two public Qwen checkpoints; no Llama approval required.


In [3]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', 'test_phm_judge_v2.py', '-v'], check=True)
subprocess.run([sys.executable, 'Files/phm_judge_v2.py', '--root', str(ROOT), '--prepare-only'], check=True)
print('Ready: 36 short judgments per model. Both candidates use identical tasks and computed facts.')


Ready: 36 short judgments per model. Both candidates use identical tasks and computed facts.


In [4]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun all cells.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'Files/phm_judge_v2.py', '--root', str(ROOT)], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('STOP: let Drive sync and send this consolidated judge report. No diagnosis outputs were changed.')


GPU: Tesla T4
Latest report: {'run_id': '20261122T061913_consolidated_grounding_judge_v2_049e5d0f', 'run_directory': 'Outputs/runs/20261122T061913_consolidated_grounding_judge_v2_049e5d0f', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T06:43:14.794576+00:00'}
STOP: let Drive sync and send this consolidated judge report. No diagnosis outputs were changed.
